In [1]:
# Imports & Load Cleaned Data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from datasets import load_dataset
import warnings
import os

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "figure.dpi": 110,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# --- Load raw data ---
print("Loading dataset...")
ds   = load_dataset(
    "kidnextdoor57/Nigerian-Financial-Transactions-and-Fraud-Detection-Dataset"
)
data = ds["train"].to_pandas()

# --- Enforce dtypes ---
data["timestamp"] = pd.to_datetime(data["timestamp"])
data["is_fraud"]  = data["is_fraud"].astype(int)

# --- Sort chronologically (CRITICAL for window features) ---
data = data.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded: {data.shape[0]:,} rows × {data.shape[1]} columns")
print(f"Date range: {data['timestamp'].min()} → {data['timestamp'].max()}")

/home/thedataboy/dev/fraud-detection-project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dataset...
Loaded: 5,000,000 rows × 45 columns
Date range: 2023-01-01 00:00:02.020872 → 2024-01-28 23:59:50.336777


In [2]:
# Drop Dead, Leaky & Redundant Features

# Every drop decision is grounded in EDA findings.
# We document the reason for each drop so this notebook
# serves as an audit trail.

DROP_FEATURES = {
    # --- LABEL LEAKAGE ---
    "fraud_type": "Direct label leak — only known AFTER fraud confirmed",

    # --- DEAD FEATURES (zero variance) ---
    "is_night_txn": "nunique=1 — same value for all 5M rows, zero information",

    # --- PERFECT MULTICOLLINEARITY (r=1.00) ---
    "user_txn_frequency_24h": "Perfect duplicate of txn_count_last_1h (r=1.0)",
    "txn_count_last_24h": "Perfect duplicate of txn_count_last_1h (r=1.0)",

    # --- HIGH MULTICOLLINEARITY (r > 0.7) ---
    "is_device_shared": "Superseded by device_seen_count (r=0.865 between them)",
    "total_amount_last_1h": "Redundant with user_avg_txn_amt (r=0.736)",

    # --- NEAR-ZERO VARIANCE / NO SIGNAL ---
    "is_ip_shared": "99.8% = 1, near-zero variance, r=-0.0002",
    "geospatial_velocity_anomaly": "Only 386 flagged cases, reversed signal",

    # --- PRE-COMPUTED SCORES NOT ALIGNED WITH EMPIRICAL RATES ---
    "channel_risk_score": "Empirical fraud rates identical across channels",
    "persona_fraud_risk": "r=0.000, all personas have identical fraud rates",
    "location_fraud_risk": "r=0.001, all cities have identical fraud rates",

    # --- ID / NON-PREDICTIVE COLUMNS ---
    "transaction_id": "Unique identifier — no predictive value",
    "ip_address": "362,384 unique values — too high cardinality for direct use",
    "device_hash": "3,835,723 unique values — too high cardinality",
    "receiver_account": "896,584 unique values — target account, not sender context",
}

print("=" * 65)
print("FEATURES BEING DROPPED")
print("=" * 65)

for feat, reason in DROP_FEATURES.items():
    exists = "✅" if feat in data.columns else "⚠️  NOT FOUND"
    print(f"\n  {exists} {feat}")
    print(f"     Reason: {reason}")

# --- Execute drops (only columns that exist) ---
cols_to_drop = [c for c in DROP_FEATURES.keys() if c in data.columns]
df = data.drop(columns=cols_to_drop).copy()

print(f"\n{'='*65}")
print(f"RESULT: {data.shape[1]} columns → {df.shape[1]} columns")
print(f"Dropped : {len(cols_to_drop)} features")
print(f"Retained: {df.shape[1]} features")

FEATURES BEING DROPPED

  ✅ fraud_type
     Reason: Direct label leak — only known AFTER fraud confirmed

  ✅ is_night_txn
     Reason: nunique=1 — same value for all 5M rows, zero information

  ✅ user_txn_frequency_24h
     Reason: Perfect duplicate of txn_count_last_1h (r=1.0)

  ✅ txn_count_last_24h
     Reason: Perfect duplicate of txn_count_last_1h (r=1.0)

  ✅ is_device_shared
     Reason: Superseded by device_seen_count (r=0.865 between them)

  ✅ total_amount_last_1h
     Reason: Redundant with user_avg_txn_amt (r=0.736)

  ✅ is_ip_shared
     Reason: 99.8% = 1, near-zero variance, r=-0.0002

  ✅ geospatial_velocity_anomaly
     Reason: Only 386 flagged cases, reversed signal

  ✅ channel_risk_score
     Reason: Empirical fraud rates identical across channels

  ✅ persona_fraud_risk
     Reason: r=0.000, all personas have identical fraud rates

  ✅ location_fraud_risk
     Reason: r=0.001, all cities have identical fraud rates

  ✅ transaction_id
     Reason: Unique identifier

In [3]:
# Fix Missing Values

# EDA Finding:
#   time_since_last_transaction → 17.93% missing
#   These NaNs are INFORMATIVE (first transaction per user)
#   Strategy: sentinel value + binary flag


# --- Check remaining missing values ---
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

if remaining_missing.empty:
    print("No missing values in retained features")
else:
    print("Missing values found:\n")
    print(remaining_missing.to_string())

Missing values found:

time_since_last_transaction    896513


In [4]:
# --- time_since_last_transaction ---
# EDA showed: missing = first-ever transaction for that account
# Meaning: NaN carries information (no prior transaction exists)

if "time_since_last_transaction" in df.columns:

    # Flag: was there a prior transaction?
    df["has_prior_transaction"] = (
        df["time_since_last_transaction"].notna().astype(int)
    )

    # Fill NaN with -1 (sentinel = "no prior transaction")
    df["time_since_last_transaction"] = (
        df["time_since_last_transaction"].fillna(-1)
    )

    print("\ntime_since_last_transaction:")
    print(f"Created: has_prior_transaction flag")
    print(f"NaN → filled with sentinel value: -1")
    print(f"has_prior_transaction distribution:")
    print(f"{df['has_prior_transaction'].value_counts().to_dict()}")


time_since_last_transaction:
Created: has_prior_transaction flag
NaN → filled with sentinel value: -1
has_prior_transaction distribution:
{1: 4103487, 0: 896513}


In [5]:
# --- time_since_last (duplicate column, similar missingness) ---
if "time_since_last" in df.columns:
    df["time_since_last"] = df["time_since_last"].fillna(-1)
    print("\ntime_since_last: NaN → -1 sentinel")


time_since_last: NaN → -1 sentinel


In [6]:
# --- avg_gap_between_txns ---
if "avg_gap_between_txns" in df.columns:
    df["avg_gap_between_txns"] = df["avg_gap_between_txns"].fillna(-1)
    print("avg_gap_between_txns: NaN → -1 sentinel")

avg_gap_between_txns: NaN → -1 sentinel


In [7]:
# --- user_std_txn_amt (first transaction has no std) ---
if "user_std_txn_amt" in df.columns:
    df["user_std_txn_amt"] = df["user_std_txn_amt"].fillna(0)
    print("user_std_txn_amt: NaN → 0 (no variation on first txn)")

user_std_txn_amt: NaN → 0 (no variation on first txn)


In [8]:
# --- Verify no remaining missing ---
final_missing = df.isnull().sum().sum()
print(f"Total remaining missing values: {final_missing}")
if final_missing == 0:
    print("All missing values resolved")

Total remaining missing values: 0
All missing values resolved


In [9]:
# --- Bool → int ---
bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
for col in bool_cols:
    df[col] = df[col].astype(int)
    print(f"{col}: bool → int")

bvn_linked: bool → int
new_device_transaction: bool → int


In [10]:
# --- sender_account: int → str (it is an ID, not a number) ---
if "sender_account" in df.columns:
    df["sender_account"] = df["sender_account"].astype(str)
    print(f"sender_account: int → str (it is an identifier)")

sender_account: int → str (it is an identifier)


In [11]:
# --- Extract temporal features from timestamp ---
# (Some were created in EDA notebook for plotting — recreate cleanly)
df["txn_hour"]       = df["timestamp"].dt.hour
df["txn_day_of_week"]= df["timestamp"].dt.dayofweek   # 0=Monday
df["txn_day_of_month"]= df["timestamp"].dt.day
df["txn_month"]      = df["timestamp"].dt.month
df["txn_is_weekend"] = (df["txn_day_of_week"] >= 5).astype(int)
df["txn_is_salary_week"] = (df["txn_day_of_month"] >= 26).astype(int)

# Replace original is_weekend and is_salary_week if they exist
# (they were pre-computed in the dataset — verify they match)
if "is_weekend" in df.columns:
    match_pct = (df["is_weekend"] == df["txn_is_weekend"]).mean() * 100
    print(f"\n  is_weekend match with recomputed: {match_pct:.2f}%")
    df.drop(columns=["is_weekend"], inplace=True)

if "is_salary_week" in df.columns:
    match_pct = (df["is_salary_week"] == df["txn_is_salary_week"]).mean() * 100
    print(f"  is_salary_week match with recomputed: {match_pct:.2f}%")
    df.drop(columns=["is_salary_week"], inplace=True)

print(f"\nTemporal features extracted cleanly from timestamp")
print(f"Current shape: {df.shape}")


  is_weekend match with recomputed: 100.00%
  is_salary_week match with recomputed: 92.00%

Temporal features extracted cleanly from timestamp
Current shape: (5000000, 34)


In [12]:
EPSILON = 1e-6  # Avoid division by zero

# ── AMOUNT CONTEXT FEATURES ─────────────────────────────────
# "Is this transaction amount unusual for this specific user?"

# Amount vs user's own average (personalised deviation)
df["amount_vs_user_avg"] = (
    df["amount_ngn"] / (df["user_avg_txn_amt"] + EPSILON)
)
print("amount_vs_user_avg = amount_ngn / user_avg_txn_amt")
print("Answers: Is this amount large for THIS user?")

amount_vs_user_avg = amount_ngn / user_avg_txn_amt
Answers: Is this amount large for THIS user?


In [13]:
# Amount deviation in standard deviation units (z-score style)
df["amount_zscore_user"] = (
    (df["amount_ngn"] - df["user_avg_txn_amt"]) /
    (df["user_std_txn_amt"] + EPSILON)
)
print("amount_zscore_user = (amount - user_mean) / user_std")
print("Answers: How many std devs from user's normal amount?")

amount_zscore_user = (amount - user_mean) / user_std
Answers: How many std devs from user's normal amount?


In [14]:
# Log-transformed amount (for autoencoder / neural networks)
df["log_amount_ngn"] = np.log1p(df["amount_ngn"])
print("log_amount_ngn = log(1 + amount_ngn)")
print("For: neural network inputs (normalises right skew)")

log_amount_ngn = log(1 + amount_ngn)
For: neural network inputs (normalises right skew)


In [15]:
# ── VELOCITY CONTEXT FEATURES ───────────────────────────────
# "Is this user transacting at an unusual rate?"

# Transaction rate acceleration
# (txn_count_last_1h vs the user's typical gap)
df["velocity_vs_typical"] = (
    df["txn_count_last_1h"] /
    (df["avg_gap_between_txns"].clip(lower=1) + EPSILON)
)
print("\nvelocity_vs_typical = txn_count_last_1h / avg_gap")
print("Answers: Is this burst of transactions unusual?")


velocity_vs_typical = txn_count_last_1h / avg_gap
Answers: Is this burst of transactions unusual?


In [16]:
# Total amount in last hour vs user average
df["hourly_amount_vs_avg"] = (
    (df["amount_ngn"] * df["txn_count_last_1h"]) /
    (df["user_avg_txn_amt"] * df["txn_count_last_1h"] + EPSILON)
)

In [17]:
# ── DEVICE CONTEXT FEATURES ─────────────────────────────────
# "New device" combined with other risk signals

# New device + high velocity (strong Account Takeover signal)
df["new_device_high_velocity"] = (
    df["new_device_transaction"].astype(int) *
    (df["txn_count_last_1h"] > df["txn_count_last_1h"].quantile(0.75))
    .astype(int)
)
print("\nnew_device_high_velocity = new_device AND high txn count")
print("Answers: New device AND transacting rapidly?")


new_device_high_velocity = new_device AND high txn count
Answers: New device AND transacting rapidly?


In [18]:
# New device + large amount vs user average
df["new_device_large_amount"] = (
    df["new_device_transaction"].astype(int) *
    (df["amount_vs_user_avg"] > 2).astype(int)
)
print("new_device_large_amount = new_device AND amount > 2x avg")
print("Answers: New device AND unusually large transaction?")

new_device_large_amount = new_device AND amount > 2x avg
Answers: New device AND unusually large transaction?


In [19]:
# Device seen count inverse (lower count = higher risk)
df["device_novelty_score"] = 1 / (df["device_seen_count"] + 1)
print("device_novelty_score = 1 / (device_seen_count + 1)")
print("Answers: How novel is this device? (1.0 = brand new)")

device_novelty_score = 1 / (device_seen_count + 1)
Answers: How novel is this device? (1.0 = brand new)


In [20]:
# ── USER HISTORY CONTEXT FEATURES ───────────────────────────

# Is this a new user? (few lifetime transactions)
df["is_new_user"] = (df["user_txn_count_total"] <= 3).astype(int)
print("\nis_new_user = user_txn_count_total <= 3")
print("Answers: Is this a newly created account?")

# Transaction count percentile within user's own history
# High count = established user, Low count = new/suspicious
df["user_maturity_score"] = np.log1p(df["user_txn_count_total"])
print("user_maturity_score = log(1 + user_txn_count_total)")
print("Answers: How established is this user account?")


is_new_user = user_txn_count_total <= 3
Answers: Is this a newly created account?
user_maturity_score = log(1 + user_txn_count_total)
Answers: How established is this user account?


In [21]:
# ── SPENDING PATTERN FEATURES ────────────────────────────────

# Spending deviation score already exists — validate and keep
if "spending_deviation_score" in df.columns:
    print("\nspending_deviation_score: pre-computed, retained")
    print(f"nunique={df['spending_deviation_score'].nunique()}")
    print(f"range=[{df['spending_deviation_score'].min():.2f}, "
          f"{df['spending_deviation_score'].max():.2f}]")


spending_deviation_score: pre-computed, retained
nunique=917
range=[-5.26, 5.02]


In [22]:
# ── TEMPORAL INTERACTION FEATURES ───────────────────────────

# Night + new device (Account Takeover at unusual hours)
# Note: since is_night_txn is dead, we use txn_hour directly
df["is_night_hour"] = (
    (df["txn_hour"] >= 23) | (df["txn_hour"] <= 5)
).astype(int)
print("\nis_night_hour: recomputed from txn_hour (11pm–5am)")
print("(replaces dead is_night_txn feature)")

df["night_new_device"] = (
    df["is_night_hour"] * df["new_device_transaction"].astype(int)
)
print("night_new_device = is_night_hour AND new_device")

# Summary
new_features = [
    "amount_vs_user_avg", "amount_zscore_user", "log_amount_ngn",
    "velocity_vs_typical", "hourly_amount_vs_avg",
    "new_device_high_velocity", "new_device_large_amount",
    "device_novelty_score", "is_new_user", "user_maturity_score",
    "is_night_hour", "night_new_device", "has_prior_transaction"
]
print(f"\n{'='*65}")
print(f"NEW FEATURES CREATED: {len(new_features)}")
for f in new_features:
    print(f"  + {f}")


is_night_hour: recomputed from txn_hour (11pm–5am)
(replaces dead is_night_txn feature)
night_new_device = is_night_hour AND new_device

NEW FEATURES CREATED: 13
  + amount_vs_user_avg
  + amount_zscore_user
  + log_amount_ngn
  + velocity_vs_typical
  + hourly_amount_vs_avg
  + new_device_high_velocity
  + new_device_large_amount
  + device_novelty_score
  + is_new_user
  + user_maturity_score
  + is_night_hour
  + night_new_device
  + has_prior_transaction


In [23]:
# Encode Categorical Features

# Different strategies for different cardinalities:
#
#   Low cardinality  (< 10 unique) → One-Hot Encoding
#   Medium cardinality (10-50)     → Target Encoding
#   High cardinality (> 50)        → Already dropped (device_hash,
#                                     ip_address, receiver_account)


cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
cat_cols = [c for c in cat_cols
            if c not in ["timestamp", "sender_account", "transaction_id"]]

print("\nCategorical columns remaining:")
for col in cat_cols:
    print(f"  {col:<30} nunique={df[col].nunique()}")

# ── ONE-HOT ENCODING (Low cardinality) ──────────────────────
OHE_COLS = [
    "payment_channel",    # 4 unique values
    "transaction_type",   # 4 unique values
    "device_used",        # 4 unique values
    "sender_persona",     # 3 unique values
    "ip_geo_region",      # 5 unique values
]
OHE_COLS = [c for c in OHE_COLS if c in df.columns]

print(f"\n{'─'*65}")
print("ONE-HOT ENCODING:")
for col in OHE_COLS:
    print(f"  {col}: {df[col].nunique()} categories")

df = pd.get_dummies(
    df,
    columns=OHE_COLS,
    drop_first=False,     # Keep all categories for interpretability
    dtype=int             # Integer not bool for XGBoost compatibility
)

print(f"OHE complete")


Categorical columns remaining:
  transaction_type               nunique=4
  merchant_category              nunique=21
  location                       nunique=10
  device_used                    nunique=4
  payment_channel                nunique=4
  sender_persona                 nunique=3
  user_top_category              nunique=21
  ip_geo_region                  nunique=5

─────────────────────────────────────────────────────────────────
ONE-HOT ENCODING:
  payment_channel: 4 categories
  transaction_type: 4 categories
  device_used: 4 categories
  sender_persona: 3 categories
  ip_geo_region: 5 categories
OHE complete


In [24]:
# ── TARGET ENCODING (Medium/High cardinality) ─────────────────
# merchant_category: 21 unique values
# location: 10 unique values
# user_top_category: 21 unique values
#
# TARGET ENCODING RULE: Must use TRAINING DATA ONLY
# We compute it here on full data for EDA purposes
# but will recompute properly on train split only
# during the training notebook to prevent leakage.

TARGET_ENCODE_COLS = [
    "merchant_category",   # 21 unique
    "location",            # 10 unique
    "user_top_category",   # 21 unique
]
TARGET_ENCODE_COLS = [c for c in TARGET_ENCODE_COLS if c in df.columns]

print("TARGET ENCODING (fraud rate per category):")
print("NOTE: Full-data target encoding shown here for EDA only.")
print("In training notebook: computed on train fold only.")

TARGET ENCODING (fraud rate per category):
NOTE: Full-data target encoding shown here for EDA only.
In training notebook: computed on train fold only.


In [25]:
# Global mean for smoothing
global_fraud_rate = df["is_fraud"].mean()
MIN_SAMPLES = 100   # Minimum samples for reliable rate estimate
SMOOTHING   = 10    # Bayesian smoothing factor

for col in TARGET_ENCODE_COLS:
    # Compute per-category fraud rate
    stats = (
        df.groupby(col)["is_fraud"]
        .agg(["sum", "count"])
        .reset_index()
    )
    stats.columns = [col, "fraud_sum", "total"]

    # Bayesian smoothing (blends category rate with global rate)
    # Prevents overfitting on rare categories
    stats["smoothed_rate"] = (
        (stats["fraud_sum"] + SMOOTHING * global_fraud_rate) /
        (stats["total"] + SMOOTHING)
    )

    # Map back to dataframe
    encode_map = dict(zip(stats[col], stats["smoothed_rate"]))
    df[f"{col}_target_enc"] = df[col].map(encode_map)

    print(f"\n{col}_target_enc")
    print(f"Categories: {df[col].nunique()}")
    print(f"Rate range: [{stats['smoothed_rate'].min():.4f}, "
          f"{stats['smoothed_rate'].max():.4f}]")
    print(f"Global fraud rate (smoothing anchor): {global_fraud_rate:.4f}")

# Drop original categorical columns after encoding
df.drop(columns=TARGET_ENCODE_COLS, inplace=True, errors="ignore")
df.drop(columns=["sender_account"], inplace=True, errors="ignore")

print(f"\n{'='*65}")
print(f"After encoding, shape: {df.shape}")


merchant_category_target_enc
Categories: 21
Rate range: [0.0349, 0.0368]
Global fraud rate (smoothing anchor): 0.0359

location_target_enc
Categories: 10
Rate range: [0.0354, 0.0364]
Global fraud rate (smoothing anchor): 0.0359

user_top_category_target_enc
Categories: 21
Rate range: [0.0343, 0.0370]
Global fraud rate (smoothing anchor): 0.0359

After encoding, shape: (5000000, 60)


In [26]:
# Temporal Train / Validation / Test Split

# WHY TEMPORAL (NOT RANDOM)?
#
# Random split on this data would cause LEAKAGE because:
#   - Window features (txn_count_last_1h, user_avg_txn_amt)
#     are computed using past transactions
#   - If future data appears in training, the model sees
#     "future" information during training
#   - This inflates validation metrics artificially
#
# Temporal split respects the time ordering.
# We simulate real deployment: train on past, test on future.

# --- Drop timestamp (not a model feature) ---
df = df.drop(columns=["timestamp"], errors="ignore")

# --- Verify is_fraud is present ---
assert "is_fraud" in df.shape or "is_fraud" in df.columns, \
    "Target column is_fraud missing!"

print("=" * 65)
print("TEMPORAL TRAIN / VAL / TEST SPLIT")
print("=" * 65)

n = len(df)
train_end = int(n * 0.75)
val_end   = int(n * 0.875)

train_df = df.iloc[:train_end].copy()
val_df   = df.iloc[train_end:val_end].copy()
test_df  = df.iloc[val_end:].copy()

# --- Separate features and target ---
TARGET = "is_fraud"
FEATURES = [c for c in df.columns if c != TARGET]

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_val   = val_df[FEATURES]
y_val   = val_df[TARGET]

X_test  = test_df[FEATURES]
y_test  = test_df[TARGET]

# --- Report ---
for name, X, y in [
    ("TRAIN",      X_train, y_train),
    ("VALIDATION", X_val,   y_val),
    ("TEST",       X_test,  y_test),
]:
    fraud_rate = y.mean() * 100
    print(f"\n  {name}")
    print(f"    Rows      : {len(X):>10,}")
    print(f"    Features  : {X.shape[1]:>10,}")
    print(f"    Fraud     : {y.sum():>10,}  ({fraud_rate:.2f}%)")
    print(f"    Legitimate: {(y==0).sum():>10,}  ({100-fraud_rate:.2f}%)")

# --- Class imbalance ratio for XGBoost ---
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\n{'='*65}")
print(f"  scale_pos_weight for XGBoost: {scale_pos_weight:.2f}")
print(f"  (n_legitimate / n_fraud in training set)")

TEMPORAL TRAIN / VAL / TEST SPLIT

  TRAIN
    Rows      :  3,750,000
    Features  :         58
    Fraud     :    134,539  (3.59%)
    Legitimate:  3,615,461  (96.41%)

  VALIDATION
    Rows      :    625,000
    Features  :         58
    Fraud     :     22,495  (3.60%)
    Legitimate:    602,505  (96.40%)

  TEST
    Rows      :    625,000
    Features  :         58
    Fraud     :     22,519  (3.60%)
    Legitimate:    602,481  (96.40%)

  scale_pos_weight for XGBoost: 26.87
  (n_legitimate / n_fraud in training set)


In [27]:
# Target encoded column names
TARGET_ENC_COLS = [
    "merchant_category_target_enc",
    "location_target_enc",
    "user_top_category_target_enc",
]

# Check if they exist in splits
existing = [c for c in TARGET_ENC_COLS if c in X_train.columns]

if existing:
    print("⚠️  Target encoded columns exist from full-data encoding.")
    print("   In production training pipeline, recompute as follows:\n")

print("""
  CORRECT PATTERN FOR TARGET ENCODING IN TRAINING:

  # Step 1: Compute on training data only
  train_merchant_rate = (
      pd.concat([X_train, y_train], axis=1)
      .groupby("merchant_category")["is_fraud"]
      .agg(["sum", "count"])
  )
  train_merchant_rate["smoothed"] = (
      (train_merchant_rate["sum"] + SMOOTHING * global_rate) /
      (train_merchant_rate["count"] + SMOOTHING)
  )
  encode_map = train_merchant_rate["smoothed"].to_dict()

  # Step 2: Apply to all splits
  X_train["merchant_enc"] = X_train["merchant_category"].map(encode_map)
  X_val["merchant_enc"]   = X_val["merchant_category"].map(encode_map)
  X_test["merchant_enc"]  = X_test["merchant_category"].map(encode_map)

  # Step 3: Fill unknown categories with global rate
  X_val["merchant_enc"].fillna(global_rate, inplace=True)
  X_test["merchant_enc"].fillna(global_rate, inplace=True)

  This will be implemented in 03_model_training.ipynb
  as part of the training pipeline.
""")

print("Noted: target encoding will be train-only in training notebook")

⚠️  Target encoded columns exist from full-data encoding.
   In production training pipeline, recompute as follows:


  CORRECT PATTERN FOR TARGET ENCODING IN TRAINING:

  # Step 1: Compute on training data only
  train_merchant_rate = (
      pd.concat([X_train, y_train], axis=1)
      .groupby("merchant_category")["is_fraud"]
      .agg(["sum", "count"])
  )
  train_merchant_rate["smoothed"] = (
      (train_merchant_rate["sum"] + SMOOTHING * global_rate) /
      (train_merchant_rate["count"] + SMOOTHING)
  )
  encode_map = train_merchant_rate["smoothed"].to_dict()

  # Step 2: Apply to all splits
  X_train["merchant_enc"] = X_train["merchant_category"].map(encode_map)
  X_val["merchant_enc"]   = X_val["merchant_category"].map(encode_map)
  X_test["merchant_enc"]  = X_test["merchant_category"].map(encode_map)

  # Step 3: Fill unknown categories with global rate
  X_val["merchant_enc"].fillna(global_rate, inplace=True)
  X_test["merchant_enc"].fillna(global_rate, inplace=True)

  Thi

In [28]:
# Feature Validation & Final Checks

# Before saving, validate:
#   1. No remaining string columns (model needs numerics)
#   2. No NaN values
#   3. No infinite values (from division in ratio features)
#   4. Target is present and binary
#   5. Feature count is reasonable

# --- 1. Check for remaining string columns ---
string_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
if string_cols:
    print(f"🚨 String columns still present: {string_cols}")
    print("   These must be encoded before training")
else:
    print("No string columns — all features are numeric")

# --- 2. Check for NaN values ---
nan_count = X_train.isnull().sum().sum()
if nan_count > 0:
    print(f"\n🚨 NaN values in X_train: {nan_count}")
    nan_cols = X_train.isnull().sum()
    print(nan_cols[nan_cols > 0].to_string())
else:
    print("No NaN values in X_train")

# --- 3. Check for infinite values ---
inf_count = np.isinf(X_train.select_dtypes(include=[np.number])).sum().sum()
if inf_count > 0:
    print(f"\n🚨 Infinite values in X_train: {inf_count}")
    inf_cols = np.isinf(X_train.select_dtypes(include=[np.number])).sum()
    print(inf_cols[inf_cols > 0].to_string())

    # Fix: replace inf with large finite value
    X_train.replace([np.inf, -np.inf], [999999, -999999], inplace=True)
    X_val.replace([np.inf, -np.inf], [999999, -999999], inplace=True)
    X_test.replace([np.inf, -np.inf], [999999, -999999], inplace=True)
    print("   Fixed: inf → 999999, -inf → -999999")
else:
    print("No infinite values in X_train")

# --- 4. Target validation ---
print(f"\nTarget column: is_fraud")
print(f"Unique values : {sorted(y_train.unique())}")
print(f"Train fraud % : {y_train.mean()*100:.2f}%")
print(f"Val   fraud % : {y_val.mean()*100:.2f}%")
print(f"Test  fraud % : {y_test.mean()*100:.2f}%")

# --- 5. Feature summary ---
print(f"\n{'='*65}")
print(f"FINAL FEATURE MATRIX SUMMARY")
print(f"{'='*65}")
print(f"  Total features : {X_train.shape[1]}")
print(f"  Training rows  : {X_train.shape[0]:,}")
print(f"  Val rows       : {X_val.shape[0]:,}")
print(f"  Test rows      : {X_test.shape[0]:,}")

# --- List all final features ---
print(f"\n  ALL {X_train.shape[1]} FEATURES:")
for i, col in enumerate(X_train.columns, 1):
    dtype = str(X_train[col].dtype)
    print(f"  {i:>3}. {col:<45} {dtype}")

No string columns — all features are numeric
No NaN values in X_train
No infinite values in X_train

Target column: is_fraud
Unique values : [np.int64(0), np.int64(1)]
Train fraud % : 3.59%
Val   fraud % : 3.60%
Test  fraud % : 3.60%

FINAL FEATURE MATRIX SUMMARY
  Total features : 58
  Training rows  : 3,750,000
  Val rows       : 625,000
  Test rows      : 625,000

  ALL 58 FEATURES:
    1. time_since_last_transaction                   float64
    2. spending_deviation_score                      float64
    3. velocity_score                                int64
    4. geo_anomaly_score                             float64
    5. amount_ngn                                    float64
    6. bvn_linked                                    int64
    7. new_device_transaction                        int64
    8. txn_hour                                      int32
    9. device_seen_count                             int64
   10. ip_seen_count                                 int64
   11. user_t

In [29]:
# Save Processed Datasets

import os
os.makedirs("../data/processed", exist_ok=True)

print("=" * 65)
print("SAVING PROCESSED DATASETS")
print("=" * 65)

# --- Save splits ---
X_train.to_parquet("../data/processed/X_train.parquet", index=False)
X_val.to_parquet("../data/processed/X_val.parquet",   index=False)
X_test.to_parquet("../data/processed/X_test.parquet",  index=False)

y_train.to_frame().to_parquet("../data/processed/y_train.parquet", index=False)
y_val.to_frame().to_parquet("../data/processed/y_val.parquet",     index=False)
y_test.to_frame().to_parquet("../data/processed/y_test.parquet",   index=False)

# --- Save feature list (critical for serving) ---
import json
feature_config = {
    "features": X_train.columns.tolist(),
    "target": "is_fraud",
    "scale_pos_weight": float(scale_pos_weight),
    "n_train": len(X_train),
    "n_val": len(X_val),
    "n_test": len(X_test),
    "train_fraud_rate": float(y_train.mean()),
    "val_fraud_rate": float(y_val.mean()),
    "test_fraud_rate": float(y_test.mean()),
}

with open("../data/processed/feature_config.json", "w") as f:
    json.dump(feature_config, f, indent=2)

print("Saved:")
print("../data/processed/X_train.parquet")
print("../data/processed/X_val.parquet")
print("../data/processed/X_test.parquet")
print("../data/processed/y_train.parquet")
print("../data/processed/y_val.parquet")
print("../data/processed/y_test.parquet")
print("../data/processed/feature_config.json")
print(f"\nFeature Engineering Complete")

SAVING PROCESSED DATASETS
Saved:
../data/processed/X_train.parquet
../data/processed/X_val.parquet
../data/processed/X_test.parquet
../data/processed/y_train.parquet
../data/processed/y_val.parquet
../data/processed/y_test.parquet
../data/processed/feature_config.json

Feature Engineering Complete
